# Caso 2 · La diferencia es pequeña. ¿Lo cerramos?
**25 minutos**

> **De:** Contraloría
> **Asunto:** Cierre de octubre
>
> Ya corrigieron el dashboard (lo del caso 1). Octubre: Comercial **$3,697,814.31**, Finanzas **$3,695,262.65**.
> La diferencia es de **$2,551.66**, menos del 0.1 %. Propongo cerrarlo como “redondeo”. ¿De acuerdo?

**Antes de abrir el código:** ¿tú lo cerrarías? Levanta la mano.

Novedades de este mes: la campaña **Hanal Pixán** (envío gratis hasta el 31 de octubre a las 23:59) y la tabla `devoluciones`.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (funciona en Colab y en local)
import importlib.util, os, sys, subprocess, pathlib
REPO_URL = "https://github.com/jona866/taller-conciliacion-de-datos-datostada2026.git"
if "google.colab" in sys.modules:
    destino = pathlib.Path("/content/taller-conciliacion-de-datos-datostada2026")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
    if any(importlib.util.find_spec(mod) is None for mod in ("duckdb", "pandas", "pytz", "pytest")):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    if not pathlib.Path("datos").exists():
        subprocess.run([sys.executable, "generar_datos.py"], check=True, stdout=subprocess.DEVNULL)
else:
    raiz = pathlib.Path.cwd()
    while not (raiz / "nnc.py").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    if not (raiz / "nnc.py").exists():
        raise FileNotFoundError("Abre el cuaderno desde el repositorio del taller.")
    os.chdir(raiz)
if not pathlib.Path("datos/manifiesto.json").exists():
    raise FileNotFoundError("Faltan los datos. Ejecuta python generar_datos.py en la raíz del repositorio.")
sys.path.insert(0, os.getcwd())
import nnc
con = nnc.conectar("caso2")
print("Listo ✅  Tablas:", ", ".join(t for (t,) in con.execute("SHOW TABLES").fetchall()))

## Paso 1 · Confirmar y acotar

In [ ]:
comercial = nnc.total(con, "caso2_dashboard_ventas")
finanzas = nnc.total(con, "caso2_reporte_finanzas")
print(f"Comercial {comercial:,.2f} · Finanzas {finanzas:,.2f} · Diferencia {comercial - finanzas:,.2f}")

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
# Compara por día, como en el caso 1 (columnas: ventas e ingreso_neto).
# Luego calcula la suma de las diferencias diarias en valor absoluto.
por_dia = None

**Diferencia pequeña no es problema pequeño.** Si la suma de diferencias diarias absolutas es mucho mayor que la diferencia neta, hay causas grandes que se están tapando entre sí.

## Paso 2 · Comparar definiciones

In [ ]:
nnc.ver_sql("caso2_dashboard_ventas")
print("=" * 70)
nnc.ver_sql("caso2_reporte_finanzas")

| | Comercial | Finanzas |
|---|---|---|
| ¿Bruto o neto? | | |
| ¿Qué fecha? | | |
| ¿En qué zona horaria? | | |

## Paso 3 · Mira los datos crudos: ¿en qué hora está `fecha_pedido`?
El diccionario dice “fecha y hora del pedido”. No dice en qué zona.

In [ ]:
horas = nnc.q(con, """
    SELECT HOUR(fecha_pedido) AS hora, canal, COUNT(*) AS pedidos
    FROM pedidos GROUP BY 1, 2
""").pivot(index="hora", columns="canal", values="pedidos").fillna(0).astype(int)
horas.T

**¿Quién compra artesanías a las 3 de la mañana?** Confírmalo con evidencia: compara la hora del pedido con la del pago, que sí trae zona.

In [ ]:
nnc.q(con, """
    SELECT p.pedido_id, p.canal, p.fecha_pedido,
           g.pagado_en AT TIME ZONE 'UTC'            AS pago_en_utc,
           g.pagado_en AT TIME ZONE 'America/Merida' AS pago_en_merida
    FROM pedidos p JOIN pagos g USING (pedido_id)
    WHERE p.fecha_pedido >= TIMESTAMP '2026-11-01 00:00:00'
    ORDER BY p.canal, p.fecha_pedido
    LIMIT 6
""")

**Conclusión:** en línea, `fecha_pedido` está en **UTC** (hora del servidor). En sucursal está en hora local. Las ventas de la noche del 31 caen en noviembre.

### ⚠️ Trampa de DuckDB (y de muchas bases)
Convertir texto **sin desplazamiento** a un tipo con zona horaria usa la zona **configurada en la sesión**. La misma consulta da resultados distintos en Colab (UTC) y en tu laptop (hora local).

In [ ]:
consulta = """
    SELECT COUNT(*) FROM pedidos
    WHERE canal = 'en_linea'
      AND CAST(CAST(fecha_pedido AS VARCHAR)::TIMESTAMPTZ AT TIME ZONE 'America/Merida' AS DATE) = DATE '2026-10-31'
"""
for zona in ["UTC", "America/Merida"]:
    con.execute(f"SET TimeZone = '{zona}'")
    print(f"Sesión en {zona:<15} → pedidos del 31 de octubre: {con.execute(consulta).fetchone()[0]}")
con.execute("SET TimeZone = 'UTC'")
print("\nForma explícita, no depende de la sesión:  (fecha_pedido AT TIME ZONE 'UTC') AT TIME ZONE 'America/Merida'")

## Pasos 4 y 5 · Cuantifica cada causa por separado
**Causa A: zona horaria.** Recalcula el dashboard con la fecha en hora de Mérida para los pedidos en línea.

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
comercial_local = None   # total del dashboard con fecha local
efecto_tz = None         # comercial - comercial_local

nnc.revisar("caso2.efecto_zona_horaria", efecto_tz)

In [ ]:
# ¿Qué pedidos cambian de mes? (los que entran desde septiembre y los que salen a noviembre)
nnc.q(con, """
    WITH x AS (
        SELECT pedido_id, fecha_pedido,
               (fecha_pedido AT TIME ZONE 'UTC') AT TIME ZONE 'America/Merida' AS fecha_local
        FROM pedidos WHERE canal = 'en_linea'
    )
    SELECT CAST(fecha_local AS DATE) AS dia_local, CAST(fecha_pedido AS DATE) AS dia_guardado, COUNT(*) AS pedidos
    FROM x
    WHERE MONTH(fecha_local) <> MONTH(fecha_pedido)
    GROUP BY 1, 2 ORDER BY 1
""")

**Causa B: definición.** Finanzas resta devoluciones; comercial no.

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
efecto_devoluciones = None

nnc.revisar("caso2.efecto_devoluciones", efecto_devoluciones)

### El puente de conciliación
Cada causa con su monto. Si queda residuo, falta una causa.

In [ ]:
nnc.puente({
    "Zona horaria (bug del dashboard)": 0,   # ← efecto_tz
    "Devoluciones (definición)": 0,          # ← efecto_devoluciones
}, comercial - finanzas)

## Paso 6 · ¿Bug o definición?
Para cada causa: ¿es un error que se arregla, o una decisión que se documenta y se acuerda con negocio? ¿Qué le contestas a Contraloría?

> ...

## Paso 7 · Volverlo prueba

In [ ]:
def dias_distintos(expr_fecha):
    return con.execute(f"""
        SELECT COUNT(*) FROM pedidos p JOIN pagos g USING (pedido_id)
        WHERE CAST({expr_fecha} AS DATE) <> CAST(g.pagado_en AT TIME ZONE 'America/Merida' AS DATE)
    """).fetchone()[0]

fecha_local = "CASE WHEN p.canal = 'en_linea' THEN (p.fecha_pedido AT TIME ZONE 'UTC') AT TIME ZONE 'America/Merida' ELSE p.fecha_pedido END"
n_crudo, n_local = dias_distintos("p.fecha_pedido"), dias_distintos(fecha_local)
nnc.probar("Fecha cruda del dashboard: el pedido cae el mismo día local que su pago", n_crudo == 0,
           f"{n_crudo} pedidos en otro día → la prueba detecta el bug")
nnc.probar("Fecha convertida a hora de Mérida: mismo día que su pago", n_local == 0, f"{n_local} pedidos")

### Lo que te llevas del caso 2
- Una diferencia pequeña puede esconder causas grandes. Mira la suma de las diferencias absolutas.
- Toda fecha sin zona es una pregunta abierta. Busca evidencia en los datos (patrón de horas, una columna que sí traiga zona).
- Separa **bug** (se arregla) de **definición** (se documenta y se decide con negocio).